In [1899]:
from pathlib import Path

import duckdb
import pandas as pd

# Phase 3 — Feature Engineering

## 1. Loading the Player-Season Dataset

### Objective

Load the validated player-season table created during the data-engineering phase and inspect the variables available for feature engineering.

Each row represents one player during the 2015/16 Premier League season.

The next steps will transform season totals into comparable performance metrics, particularly statistics per 90 minutes.

### Important Principle

Only variables that exist in the validated source dataset will be used. Similarity modeling will begin only after the feature definitions have been tested and interpreted.

In [1900]:
from pathlib import Path

import pandas as pd


# Locate the project root.
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Define the validated input file from phase 2.
PLAYER_SEASON_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "player_season.parquet"
)


# Load the season-level player table.
player_season = pd.read_parquet(
    PLAYER_SEASON_PATH
)

print(f"Dataset shape: {player_season.shape}")

Dataset shape: (550, 24)


## 2. Inspecting the Source Schema

### Objective

Inspect the columns and data types available in the player-season table before defining new features.

This step ensures that feature engineering uses only variables that are present in the validated dataset.

In [1901]:
schema_summary = pd.DataFrame({
    "column": player_season.columns,
    "dtype": player_season.dtypes.astype(str).values,
})

schema_summary

,column,dtype
0,competition_id,int64
1,season_id,int64
2,player_id,int64
3,player_name,str
4,primary_position_id,int64
5,primary_position,str
6,appearances,int32
7,starts,int32
8,minutes_played,float64
9,passes_attempted,int32


## 3. Creating Per-90 Metrics

### Objective

Convert season totals into rates per 90 minutes so that players with different amounts of playing time can be compared more fairly.

### Formula

For an event count \(x\):

\[
x_{per90} = \frac{x}{minutes\_played} \times 90
\]

Per-90 metrics describe how frequently a player performs an action during the equivalent of one full match.

### Limitation

Per-90 normalization does not eliminate the uncertainty associated with players who have played very few minutes. A minimum playing-time rule will therefore be introduced later.

In [1902]:
# Preserve the validated phase-2 table.
player_features = player_season.copy()


# Per-90 calculations require positive playing time.
assert (
    player_features["minutes_played"] > 0
).all()


# Calculate the first per-90 feature.
player_features["passes_attempted_per90"] = (
    player_features["passes_attempted"]
    / player_features["minutes_played"]
    * 90
)

In [1903]:
player_features[[
    "player_name",
    "minutes_played",
    "passes_attempted",
    "passes_attempted_per90",
]].head(10)

,player_name,minutes_played,passes_attempted,passes_attempted_per90
0,Kasper Schmeichel,3594.016667,1220,30.550777
1,Wes Morgan,3594.016667,849,21.260335
2,Simon Francis,3591.566667,2454,61.494056
3,Andrew Surman,3590.933333,2309,57.870749
4,Toby Alderweireld,3569.183333,2122,53.508039
5,Heurelho da Silva Gomes,3554.150000,1132,28.665082
6,Craig Dawson,3548.666667,1658,42.049596
7,Darren Fletcher,3533.583333,1516,38.612362
8,Łukasz Fabiański,3492.200000,1212,31.235324
9,Harry Kane,3486.650000,1005,25.941807


### Applying Per-90 Normalization to Event Volumes

The validated player-season table contains several cumulative event statistics.

The same per-90 formula is applied to each volume variable. Appearances, starts, identifiers and positions are excluded because they are contextual variables rather than event volumes.

In [1904]:
# Season totals that depend directly on playing time.
volume_columns = [
    "passes_attempted",
    "passes_completed",
    "shots",
    "goals",
    "total_xg",
    "non_penalty_xg",
    "carries",
    "dribbles_attempted",
    "dribbles_completed",
    "pressures",
    "counterpressures",
    "interceptions",
    "ball_recoveries",
    "clearances",
    "blocks",
]


# Apply the same normalization to every volume.
for column in volume_columns:
    player_features[f"{column}_per90"] = (
        player_features[column]
        / player_features["minutes_played"]
        * 90
    )

In [1905]:
per90_columns = [
    f"{column}_per90"
    for column in volume_columns
]

assert (
    player_features[per90_columns]
    .notna()
    .all()
    .all()
)

assert (
    player_features[per90_columns] >= 0
).all().all()

print(
    f"{len(per90_columns)} per-90 features "
    "created and validated."
)

15 per-90 features created and validated.


## 4. Creating Success and Style Rates

### Objective

Complement event-volume features with ratios that describe execution success and playing style.

The following rates are created:

- Pass completion rate.
- Dribble completion rate.
- Counterpressures as a share of total pressures.

A rate is left missing when its denominator is zero because no attempts and zero success are not the same situation.

In [1906]:
assert (
    player_features["passes_completed"]
    <= player_features["passes_attempted"]
).all()

assert (
    player_features["dribbles_completed"]
    <= player_features["dribbles_attempted"]
).all()

assert (
    player_features["counterpressures"]
    <= player_features["pressures"]
).all()

print("Numerator and denominator checks passed.")

Numerator and denominator checks passed.


In [1907]:
def calculate_rate(numerator, denominator):
    """
    Divide two columns while preserving missing rates
    when the denominator is zero.
    """

    valid_denominator = denominator.where(
        denominator > 0
    )

    return numerator / valid_denominator

In [1908]:
player_features["pass_completion_rate"] = (
    calculate_rate(
        player_features["passes_completed"],
        player_features["passes_attempted"],
    )
)

player_features["dribble_completion_rate"] = (
    calculate_rate(
        player_features["dribbles_completed"],
        player_features["dribbles_attempted"],
    )
)

player_features["counterpressure_share"] = (
    calculate_rate(
        player_features["counterpressures"],
        player_features["pressures"],
    )
)

In [1909]:
rate_columns = [
    "pass_completion_rate",
    "dribble_completion_rate",
    "counterpressure_share",
]

for column in rate_columns:
    available_values = (
        player_features[column]
        .dropna()
    )

    assert available_values.between(
        0,
        1,
    ).all()

print("Rate features validated.")

player_features[
    rate_columns
].isna().sum()

Rate features validated.


pass_completion_rate        5
dribble_completion_rate    80
counterpressure_share      25
dtype: int64

## 5. Controlling for Small Playing-Time Samples

### Objective

Identify players with enough playing time to produce reasonably stable season-level features.

Players with fewer than 900 minutes are retained in the dataset but are not initially considered eligible for similarity comparisons.

### Rationale

Per-90 normalization makes playing time comparable, but it does not make small samples reliable. A player with only a few minutes can have extreme per-90 values based on very few actions.

In [1910]:
MINIMUM_MINUTES = 900

player_features["eligible_for_similarity"] = (
    player_features["minutes_played"]
    >= MINIMUM_MINUTES
)

In [1911]:
eligibility_summary = (
    player_features["eligible_for_similarity"]
    .value_counts()
    .rename_axis("eligible_for_similarity")
    .reset_index(name="players")
)

eligibility_summary

,eligible_for_similarity,players
0,True,327
1,False,223


In [1912]:
eligible_players = player_features.loc[
    player_features["eligible_for_similarity"]
]

print(
    "Eligible players:",
    len(eligible_players),
)

eligible_players[
    rate_columns
].isna().sum()

Eligible players: 327


pass_completion_rate        0
dribble_completion_rate    17
counterpressure_share       1
dtype: int64

## 6. Inspecting Player Positions

### Objective

Inspect the StatsBomb positions represented in the player-season dataset before grouping players into broader comparison roles.

### Rationale

Players should primarily be compared with players performing similar tactical functions. However, the original StatsBomb positions must be examined before defining broader role categories.

In [1913]:
position_summary = (
    player_features
    .groupby(
        [
            "primary_position_id",
            "primary_position",
        ],
        as_index=False,
    )
    .agg(
        players=(
            "player_id",
            "nunique",
        ),
        eligible_players=(
            "eligible_for_similarity",
            "sum",
        ),
        total_minutes=(
            "minutes_played",
            "sum",
        ),
    )
    .sort_values(
        "primary_position_id"
    )
)

position_summary

,primary_position_id,primary_position,players,eligible_players,total_minutes
0,1,Goalkeeper,47,27,71765.216667
1,2,Right Back,50,28,70354.383333
2,3,Right Center Back,44,32,81068.716667
3,5,Left Center Back,37,28,71124.116667
4,6,Left Back,46,31,72964.266667
5,7,Right Wing Back,1,0,157.366667
6,9,Right Defensive Midfield,21,14,34889.383333
7,10,Center Defensive Midfield,11,6,12660.066667
8,11,Left Defensive Midfield,24,17,41762.750000
9,12,Right Midfield,18,8,21168.483333


## 7. Creating Broad Player Roles

### Objective

Group detailed StatsBomb positions into broader tactical roles for role-aware player comparisons.

### Design Choices

Left-sided and right-sided variants are grouped together because laterality should not define a separate similarity space.

The original StatsBomb position is preserved. The broader role is an additional variable used to compare players with reasonably similar tactical responsibilities.

In [1914]:
role_by_position_id = {
    # Goalkeeper
    1: "goalkeeper",

    # Full backs and wing backs
    2: "full_back",
    6: "full_back",
    7: "full_back",

    # Centre backs
    3: "centre_back",
    5: "centre_back",

    # Defensive midfielders
    9: "defensive_midfielder",
    10: "defensive_midfielder",
    11: "defensive_midfielder",

    # Central midfielders
    13: "central_midfielder",
    15: "central_midfielder",

    # Wide midfielders and wingers
    12: "wide_player",
    16: "wide_player",
    17: "wide_player",
    21: "wide_player",

    # Attacking midfielders
    18: "attacking_midfielder",
    19: "attacking_midfielder",

    # Forwards
    22: "forward",
    23: "forward",
    24: "forward",
}

In [1915]:
player_features["role"] = (
    player_features["primary_position_id"]
    .map(role_by_position_id)
)

In [1916]:
unmapped_positions = (
    player_features.loc[
        player_features["role"].isna(),
        [
            "primary_position_id",
            "primary_position",
        ],
    ]
    .drop_duplicates()
)

assert unmapped_positions.empty

print("All observed positions were assigned to a role.")

All observed positions were assigned to a role.


In [1917]:
role_summary = (
    player_features
    .groupby(
        "role",
        as_index=False,
    )
    .agg(
        players=(
            "player_id",
            "nunique",
        ),
        eligible_players=(
            "eligible_for_similarity",
            "sum",
        ),
        total_minutes=(
            "minutes_played",
            "sum",
        ),
    )
    .sort_values(
        "eligible_players",
        ascending=False,
    )
)

assert role_summary["players"].sum() == len(
    player_features
)

assert role_summary["eligible_players"].sum() == len(
    eligible_players
)

role_summary

,role,players,eligible_players,total_minutes
2,centre_back,81,60,152192.833333
7,wide_player,111,60,127402.033333
5,full_back,97,59,143476.016667
3,defensive_midfielder,56,37,89312.200000
4,forward,77,36,86028.083333
1,central_midfielder,49,31,73582.400000
6,goalkeeper,47,27,71765.216667
0,attacking_midfielder,32,17,45092.600000


### Validating Missing Rates by Role

Missing success rates are examined within each role.

This helps distinguish genuine data problems from actions that are uncommon or irrelevant for particular tactical roles.

In [1918]:
eligible_role_players = player_features.loc[
    player_features["eligible_for_similarity"]
].copy()

missing_rates_by_role = (
    eligible_role_players
    .groupby("role")[rate_columns]
    .agg(
        lambda values: values.isna().sum()
    )
)

missing_rates_by_role

,pass_completion_rate,dribble_completion_rate,counterpressure_share
role,,,
attacking_midfielder,0,0,0
central_midfielder,0,0,0
centre_back,0,1,0
defensive_midfielder,0,0,0
forward,0,0,0
full_back,0,0,0
goalkeeper,0,16,1
wide_player,0,0,0


## 8. Building Event-Level Passing Features

### Objective

Use the complete StatsBomb event dataset to create passing-style features that are not available in the player-season table.

The first event-derived features will describe:

- Pass length.
- Forward ball progression.
- Progressive passing.

DuckDB is used to aggregate the Parquet files directly without loading the complete event dataset into memory.

In [1919]:
EVENTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "events"
)

event_files = sorted(
    EVENTS_DIR.glob("*.parquet")
)

EVENTS_GLOB = (
    EVENTS_DIR
    / "*.parquet"
).as_posix()

assert len(event_files) == 380

print(
    f"Processed event files: "
    f"{len(event_files)}"
)

Processed event files: 380


In [1920]:
event_schema = duckdb.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{EVENTS_GLOB}',
        union_by_name = true
    )
""").df()

required_pass_columns = [
    "player_id",
    "player",
    "is_pass",
    "pass_completed",
    "pass_length",
    "location",
    "pass_end_location",
]

available_event_columns = set(
    event_schema["column_name"]
)

missing_pass_columns = [
    column
    for column in required_pass_columns
    if column not in available_event_columns
]

assert not missing_pass_columns

event_schema.loc[
    event_schema["column_name"].isin(
        required_pass_columns
    ),
    [
        "column_name",
        "column_type",
        "null",
    ],
]

,column_name,column_type,null
13,player_id,DOUBLE,YES
14,player,VARCHAR,YES
17,location,DOUBLE[],YES
23,pass_end_location,DOUBLE[],YES
24,pass_length,DOUBLE,YES
62,is_pass,BOOLEAN,YES
63,pass_completed,BOOLEAN,YES


### Pass-Length Features

Pass length describes a player's preferred distribution range.

Two features are calculated:

- `average_pass_length`: Average length of all attempted passes.
- `average_completed_pass_length`: Average length of completed passes.

These features are averages rather than per-90 metrics because they describe the typical characteristics of an individual pass, not the frequency of an action.

In [1921]:
pass_length_features = duckdb.sql(f"""
    SELECT
        CAST(player_id AS BIGINT)
            AS player_id,

        ANY_VALUE(player)
            AS player_name_events,

        COUNT(*)
            AS event_passes_attempted,

        SUM(
            CASE
                WHEN pass_completed THEN 1
                ELSE 0
            END
        )
            AS event_passes_completed,

        COUNT(pass_length)
            AS passes_with_length,

        AVG(pass_length)
            AS average_pass_length,

        AVG(
            CASE
                WHEN pass_completed
                THEN pass_length
            END
        )
            AS average_completed_pass_length

    FROM read_parquet(
        '{EVENTS_GLOB}',
        union_by_name = true
    )

    WHERE
        is_pass
        AND player_id IS NOT NULL

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1922]:
assert (
    pass_length_features[
        "passes_with_length"
    ].sum()
    == pass_length_features[
        "event_passes_attempted"
    ].sum()
)

assert (
    pass_length_features[
        "event_passes_attempted"
    ].sum()
    == player_features[
        "passes_attempted"
    ].sum()
)

assert (
    pass_length_features[
        "event_passes_completed"
    ].sum()
    == player_features[
        "passes_completed"
    ].sum()
)

print(
    "Pass counts and pass lengths "
    "validated."
)

Pass counts and pass lengths validated.


In [1923]:
pass_length_features[[
    "player_name_events",
    "event_passes_attempted",
    "average_pass_length",
    "average_completed_pass_length",
]].sort_values(
    "event_passes_attempted",
    ascending=False,
).head(10)

,player_name_events,event_passes_attempted,average_pass_length,average_completed_pass_length
118,Francesc Fàbregas i Soler,2947,20.542909,19.333113
129,Mesut Özil,2511,17.514368,16.416553
175,Simon Francis,2454,22.726326,20.570361
80,Andrew Surman,2309,21.591837,20.713837
130,Ignacio Monreal Eraso,2259,17.255343,15.963730
81,Charlie Daniels,2239,19.321910,17.210017
63,Daley Blind,2215,24.803332,23.018692
237,César Azpilicueta Tanco,2198,18.896098,17.572822
138,Aaron Ramsey,2197,18.350989,17.861463
451,Toby Alderweireld,2122,28.365929,25.613836


### Merging Pass-Length Features

The validated pass-length features are merged into the player feature table using the unique StatsBomb player identifier.

Event-level pass counts are not duplicated because the player-season table already contains the corresponding totals.

In [1924]:
pass_length_columns = [
    "player_id",
    "average_pass_length",
    "average_completed_pass_length",
]

assert (
    pass_length_features["player_id"]
    .is_unique
)

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        pass_length_features[
            pass_length_columns
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

In [1925]:
assert len(player_features) == rows_before_merge

eligible_pass_length_missing = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        [
            "average_pass_length",
            "average_completed_pass_length",
        ],
    ]
    .isna()
    .sum()
)

assert (
    eligible_pass_length_missing == 0
).all()

print(
    "Pass-length features merged "
    "and validated."
)

Pass-length features merged and validated.


In [1926]:
player_features.loc[
    player_features["eligible_for_similarity"],
    [
        "player_name",
        "role",
        "passes_attempted_per90",
        "average_pass_length",
        "average_completed_pass_length",
    ],
].sort_values(
    "average_pass_length",
    ascending=False,
).head(10)

,player_name,role,passes_attempted_per90,average_pass_length,average_completed_pass_length
0,Kasper Schmeichel,goalkeeper,30.550777,63.403215,52.035058
256,Ben Foster,goalkeeper,26.025633,60.197937,52.361350
155,Glyn Oliver Myhill,goalkeeper,27.329316,59.194738,50.892871
92,Wayne Hennessey,goalkeeper,28.578737,58.001231,45.668569
224,Fraser Forster,goalkeeper,25.755587,57.684659,49.964223
217,Vito Mannone,goalkeeper,31.707837,57.346413,45.513229
5,Heurelho da Silva Gomes,goalkeeper,28.665082,57.185207,48.037622
314,Declan Rudd,goalkeeper,28.991657,55.813622,44.624055
323,Mark Bunn,goalkeeper,25.820391,55.626908,46.951016
188,Rob Elliot,goalkeeper,31.220168,54.329198,46.574177


### Forward Pass Progression

Pass length measures total distance but does not indicate direction.

Forward progression is measured using the change in the StatsBomb x-coordinate:

\[
delta\_x = end\_x - start\_x
\]

A positive value moves the ball toward the opponent's goal, while a negative value moves it toward the player's own goal.

StatsBomb normalizes attacking direction, so the interpretation is consistent across teams and match periods.

In [1927]:
pass_progression_features = duckdb.sql(f"""
    WITH pass_locations AS (
        SELECT
            CAST(player_id AS BIGINT)
                AS player_id,

            pass_completed,

            pass_end_location[1]
            - location[1]
                AS delta_x

        FROM read_parquet(
            '{EVENTS_GLOB}',
            union_by_name = true
        )

        WHERE
            is_pass
            AND player_id IS NOT NULL
            AND location IS NOT NULL
            AND pass_end_location IS NOT NULL
    )

    SELECT
        player_id,

        COUNT(*)
            AS spatial_passes,

        AVG(delta_x)
            AS average_pass_progression,

        AVG(
            CASE
                WHEN pass_completed
                THEN delta_x
            END
        )
            AS average_completed_pass_progression,

        SUM(
            CASE
                WHEN
                    pass_completed
                    AND delta_x > 0
                THEN delta_x
                ELSE 0
            END
        )
            AS completed_forward_distance

    FROM pass_locations

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1928]:
assert (
    pass_progression_features[
        "spatial_passes"
    ].sum()
    == player_features[
        "passes_attempted"
    ].sum()
)

assert (
    pass_progression_features[
        "player_id"
    ].is_unique
)

print(
    "All pass events contain valid "
    "spatial coordinates."
)

All pass events contain valid spatial coordinates.


In [1929]:
pass_progression_features.sort_values(
    "completed_forward_distance",
    ascending=False,
).head(10)

,player_id,spatial_passes,average_pass_progression,average_completed_pass_progression,completed_forward_distance
141,3522,1132,52.574823,41.929005,24885.6
105,3461,1036,44.312934,33.212691,21746.1
226,3815,1220,58.198361,44.767715,21354.2
38,3262,1212,38.172195,26.296745,20203.9
118,3478,2947,7.560027,5.544555,20071.9
63,3311,2215,10.655892,8.227324,19303.2
182,3623,998,46.090481,34.184173,19006.4
150,3537,1013,47.400296,34.032668,18760.1
29,3099,1217,34.615119,22.682987,18546.8
417,12608,840,46.779762,37.631893,18311.0


### Merging Pass-Progression Features

Average progression describes the typical direction of an individual pass and therefore does not require per-90 normalization.

Completed forward distance is a cumulative season total. It is converted to a per-90 metric before being used for player comparisons.

In [1930]:
progression_columns = [
    "player_id",
    "average_pass_progression",
    "average_completed_pass_progression",
    "completed_forward_distance",
]

assert (
    pass_progression_features["player_id"]
    .is_unique
)

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        pass_progression_features[
            progression_columns
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(player_features) == rows_before_merge

In [1931]:
player_features[
    "completed_forward_distance_per90"
] = (
    player_features[
        "completed_forward_distance"
    ]
    / player_features["minutes_played"]
    * 90
)

In [1932]:
players_with_passes = (
    player_features[
        "average_pass_length"
    ].notna()
)

assert (
    player_features.loc[
        players_with_passes,
        "average_pass_progression",
    ].abs()
    <= player_features.loc[
        players_with_passes,
        "average_pass_length",
    ] + 1e-9
).all()

assert (
    player_features.loc[
        players_with_passes,
        "average_completed_pass_progression",
    ].abs()
    <= player_features.loc[
        players_with_passes,
        "average_completed_pass_length",
    ] + 1e-9
).all()

assert (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        "completed_forward_distance_per90",
    ].notna()
).all()

print(
    "Pass-progression features merged "
    "and validated."
)

Pass-progression features merged and validated.


In [1933]:
player_features.loc[
    player_features["eligible_for_similarity"],
    [
        "player_name",
        "role",
        "passes_attempted_per90",
        "average_pass_progression",
        "average_completed_pass_progression",
        "completed_forward_distance_per90",
    ],
].sort_values(
    "completed_forward_distance_per90",
    ascending=False,
).head(15)

,player_name,role,passes_attempted_per90,average_pass_progression,average_completed_pass_progression,completed_forward_distance_per90
188,Rob Elliot,goalkeeper,31.220168,48.955959,39.722251,688.849412
5,Heurelho da Silva Gomes,goalkeeper,28.665082,52.574823,41.929005,630.165862
103,Bradley Guzan,goalkeeper,28.507325,46.779762,37.631893,621.425743
40,Petr Čech,goalkeeper,29.160433,44.312934,33.212691,612.090446
113,John Ruddy,goalkeeper,31.951200,46.673481,34.354365,611.295308
224,Fraser Forster,goalkeeper,25.755587,53.569815,44.889844,607.757820
256,Ben Foster,goalkeeper,26.025633,56.045833,46.820297,603.354557
61,Francesc Fàbregas i Soler,defensive_midfielder,87.319956,7.560027,5.544555,594.732753
155,Glyn Oliver Myhill,goalkeeper,27.329316,54.781089,45.283871,581.345410
235,Costel Fane Pantilimon,goalkeeper,31.434185,48.623393,36.700356,578.882964


### Progressive Passes

There is no universal definition of a progressive pass. This project uses a transparent spatial definition.

A completed pass is considered progressive when its end location is at least 25% closer to the centre of the opponent's goal than its start location.

The opponent's goal centre is represented by the StatsBomb coordinate `(120, 40)`.

Formally:

\[
end\_goal\_distance \leq 0.75 \times start\_goal\_distance
\]

The 25% threshold prevents very small forward movements from being classified as progressive.

In [1934]:
PROGRESSIVE_DISTANCE_RATIO = 0.75

In [1935]:
progressive_pass_features = duckdb.sql(f"""
    WITH pass_distances AS (
        SELECT
            CAST(player_id AS BIGINT)
                AS player_id,

            player,

            pass_completed,

            SQRT(
                POWER(
                    120.0 - location[1],
                    2
                )
                +
                POWER(
                    40.0 - location[2],
                    2
                )
            )
                AS start_goal_distance,

            SQRT(
                POWER(
                    120.0
                    - pass_end_location[1],
                    2
                )
                +
                POWER(
                    40.0
                    - pass_end_location[2],
                    2
                )
            )
                AS end_goal_distance

        FROM read_parquet(
            '{EVENTS_GLOB}',
            union_by_name = true
        )

        WHERE
            is_pass
            AND player_id IS NOT NULL
            AND location IS NOT NULL
            AND pass_end_location IS NOT NULL
    ),

    classified_passes AS (
        SELECT
            *,

            (
                pass_completed
                AND end_goal_distance
                    <= {PROGRESSIVE_DISTANCE_RATIO}
                    * start_goal_distance
            )
                AS is_progressive_pass

        FROM pass_distances
    )

    SELECT
        player_id,

        ANY_VALUE(player)
            AS player_name_events,

        SUM(
            CASE
                WHEN pass_completed THEN 1
                ELSE 0
            END
        )
            AS completed_passes,

        SUM(
            CASE
                WHEN is_progressive_pass THEN 1
                ELSE 0
            END
        )
            AS progressive_passes

    FROM classified_passes

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1936]:
assert (
    progressive_pass_features[
        "progressive_passes"
    ]
    <= progressive_pass_features[
        "completed_passes"
    ]
).all()

assert (
    progressive_pass_features[
        "completed_passes"
    ].sum()
    == player_features[
        "passes_completed"
    ].sum()
)

assert (
    progressive_pass_features[
        "player_id"
    ].is_unique
)

print(
    "Progressive passes classified "
    "and validated."
)

Progressive passes classified and validated.


In [1937]:
progressive_pass_features.sort_values(
    "progressive_passes",
    ascending=False,
).head(15)

,player_id,player_name_events,completed_passes,progressive_passes
118,3478,Francesc Fàbregas i Soler,2406.0,368.0
129,3496,Mesut Özil,2050.0,358.0
141,3522,Heurelho da Silva Gomes,593.0,352.0
223,3812,Christian Fuchs,1128.0,316.0
226,3815,Kasper Schmeichel,477.0,314.0
142,3523,Craig Dawson,1049.0,305.0
63,3311,Daley Blind,1786.0,304.0
237,3957,César Azpilicueta Tanco,1782.0,283.0
105,3461,Petr Čech,654.0,270.0
175,3608,Simon Francis,1921.0,267.0


### Merging Progressive-Pass Features

Progressive-pass totals are merged into the player feature table and normalized per 90 minutes.

Players with no attempted passes receive a progressive-pass count of zero. The raw count is preserved for validation, while the per-90 version will be used for comparisons.

In [1938]:
progressive_columns = [
    "player_id",
    "progressive_passes",
]

assert (
    progressive_pass_features["player_id"]
    .is_unique
)

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        progressive_pass_features[
            progressive_columns
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(player_features) == rows_before_merge

In [1939]:
player_features["progressive_passes"] = (
    player_features["progressive_passes"]
    .fillna(0)
    .astype("int32")
)

In [1940]:
player_features[
    "progressive_passes_per90"
] = (
    player_features["progressive_passes"]
    / player_features["minutes_played"]
    * 90
)

In [1941]:
assert (
    player_features["progressive_passes"]
    <= player_features["passes_completed"]
).all()

assert (
    player_features[
        "progressive_passes_per90"
    ] >= 0
).all()

assert (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        "progressive_passes_per90",
    ].notna()
).all()

print(
    "Progressive-pass features merged "
    "and validated."
)

Progressive-pass features merged and validated.


In [1942]:
progression_leaders_by_role = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        [
            "player_name",
            "primary_position",
            "role",
            "passes_attempted_per90",
            "progressive_passes_per90",
            "average_completed_pass_progression",
        ],
    ]
    .sort_values(
        [
            "role",
            "progressive_passes_per90",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "role",
        group_keys=False,
    )
    .head(3)
)

progression_leaders_by_role

,player_name,primary_position,role,passes_attempted_per90,progressive_passes_per90,average_completed_pass_progression
47,Mesut Özil,Center Attacking Midfield,attacking_midfielder,72.079993,10.276638,1.523512
96,Steven Davis,Center Attacking Midfield,attacking_midfielder,54.508052,6.771808,2.154077
211,David Josué Jiménez Silva,Center Attacking Midfield,attacking_midfielder,72.838689,6.706250,2.587765
174,Michael Carrick,Right Center Midfield,central_midfielder,68.274757,8.479707,5.140798
115,James Philip Milner,Right Center Midfield,central_midfielder,57.080953,8.331692,2.400853
266,Jordan Brian Henderson,Right Center Midfield,central_midfielder,67.494973,7.037708,3.754359
32,Daley Blind,Left Center Back,centre_back,61.420355,8.429701,8.227324
2,Simon Francis,Right Center Back,centre_back,61.494056,6.690674,3.984071
84,Russell Martin,Right Center Back,centre_back,42.359925,5.883323,7.054779
61,Francesc Fàbregas i Soler,Right Defensive Midfield,defensive_midfielder,87.319956,10.903883,5.544555


## 9. Evaluating Passing-Creativity Features

### Objective

Evaluate candidate passing features before adding them to the final player table.

The candidate variables describe:

- Passes leading directly to a shot.
- Goal assists.
- Crosses.
- Switches of play.
- Through balls.

Not every available variable should automatically become a model feature. Event frequency and player coverage are inspected first.

In [1943]:
creativity_source_columns = [
    "pass_shot_assist",
    "pass_goal_assist",
    "pass_cross",
    "pass_switch",
    "pass_through_ball",
]

missing_creativity_columns = [
    column
    for column in creativity_source_columns
    if column not in available_event_columns
]

assert not missing_creativity_columns

print(
    "Creativity source columns "
    "are available."
)

Creativity source columns are available.


In [1944]:
creativity_candidates = duckdb.sql(f"""
    WITH normalized_pass_flags AS (
        SELECT
            CAST(player_id AS BIGINT)
                AS player_id,

            player,

            COALESCE(
                TRY_CAST(
                    pass_shot_assist
                    AS BOOLEAN
                ),
                FALSE
            )
                AS is_shot_assist,

            COALESCE(
                TRY_CAST(
                    pass_goal_assist
                    AS BOOLEAN
                ),
                FALSE
            )
                AS is_goal_assist,

            COALESCE(
                TRY_CAST(
                    pass_cross
                    AS BOOLEAN
                ),
                FALSE
            )
                AS is_cross,

            COALESCE(
                TRY_CAST(
                    pass_switch
                    AS BOOLEAN
                ),
                FALSE
            )
                AS is_switch,

            COALESCE(
                TRY_CAST(
                    pass_through_ball
                    AS BOOLEAN
                ),
                FALSE
            )
                AS is_through_ball

        FROM read_parquet(
            '{EVENTS_GLOB}',
            union_by_name = true
        )

        WHERE
            is_pass
            AND player_id IS NOT NULL
    )

    SELECT
        player_id,

        ANY_VALUE(player)
            AS player_name_events,

        SUM(
            CASE
                WHEN
                    is_shot_assist
                    OR is_goal_assist
                THEN 1
                ELSE 0
            END
        )
            AS key_passes,

        SUM(
            CASE
                WHEN is_goal_assist THEN 1
                ELSE 0
            END
        )
            AS goal_assists,

        SUM(
            CASE
                WHEN is_cross THEN 1
                ELSE 0
            END
        )
            AS crosses,

        SUM(
            CASE
                WHEN is_switch THEN 1
                ELSE 0
            END
        )
            AS switches,

        SUM(
            CASE
                WHEN is_through_ball THEN 1
                ELSE 0
            END
        )
            AS through_balls

    FROM normalized_pass_flags

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1945]:
creativity_evaluation = (
    creativity_candidates
    .merge(
        player_features[[
            "player_id",
            "role",
            "minutes_played",
            "eligible_for_similarity",
        ]],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

In [1946]:
candidate_columns = [
    "key_passes",
    "goal_assists",
    "crosses",
    "switches",
    "through_balls",
]

eligible_creativity = (
    creativity_evaluation.loc[
        creativity_evaluation[
            "eligible_for_similarity"
        ]
    ]
)

creativity_summary = pd.DataFrame({
    "feature": candidate_columns,

    "total_events": [
        creativity_evaluation[
            column
        ].sum()
        for column in candidate_columns
    ],

    "eligible_players_with_event": [
        (
            eligible_creativity[
                column
            ] > 0
        ).sum()
        for column in candidate_columns
    ],

    "eligible_player_coverage": [
        (
            eligible_creativity[
                column
            ] > 0
        ).mean()
        for column in candidate_columns
    ],
})

creativity_summary

,feature,total_events,eligible_players_with_event,eligible_player_coverage
0,key_passes,7175.0,308,0.941896
1,goal_assists,669.0,203,0.620795
2,crosses,9545.0,284,0.868502
3,switches,10488.0,325,0.993884
4,through_balls,1621.0,262,0.801223


### Selecting Creativity Features

Four passing-creativity variables are retained:

- Key passes.
- Crosses.
- Switches of play.
- Through balls.

Goal assists are excluded from the similarity feature set because they are outcome-dependent and already represented within key passes. Including both variables would partially duplicate the same information.

In [1947]:
assert (
    creativity_candidates["goal_assists"]
    <= creativity_candidates["key_passes"]
).all()

print(
    "Every goal assist is included "
    "within the key-pass definition."
)

Every goal assist is included within the key-pass definition.


In [1948]:
selected_creativity_columns = [
    "key_passes",
    "crosses",
    "switches",
    "through_balls",
]

creativity_columns_to_merge = [
    "player_id",
    *selected_creativity_columns,
]

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        creativity_candidates[
            creativity_columns_to_merge
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(player_features) == rows_before_merge

In [1949]:
for column in selected_creativity_columns:
    player_features[column] = (
        player_features[column]
        .fillna(0)
        .astype("int32")
    )

In [1950]:
creativity_per90_columns = []

for column in selected_creativity_columns:
    per90_column = f"{column}_per90"

    player_features[per90_column] = (
        player_features[column]
        / player_features["minutes_played"]
        * 90
    )

    creativity_per90_columns.append(
        per90_column
    )

In [1951]:
for column in selected_creativity_columns:
    assert (
        player_features[column]
        <= player_features[
            "passes_attempted"
        ]
    ).all()

assert (
    player_features[
        creativity_per90_columns
    ] >= 0
).all().all()

assert (
    player_features[
        creativity_per90_columns
    ].notna()
).all().all()

print(
    "Creativity features merged "
    "and validated."
)

Creativity features merged and validated.


In [1952]:
player_features.loc[
    player_features["eligible_for_similarity"],
    [
        "player_name",
        "primary_position",
        "role",
        "key_passes_per90",
        "crosses_per90",
        "switches_per90",
        "through_balls_per90",
    ],
].sort_values(
    "key_passes_per90",
    ascending=False,
).head(15)

,player_name,primary_position,role,key_passes_per90,crosses_per90,switches_per90,through_balls_per90
47,Mesut Özil,Center Attacking Midfield,attacking_midfielder,4.076208,2.554807,2.095516,0.918582
98,Dimitri Payet,Left Wing,wide_player,3.750558,4.185891,3.181277,0.904152
172,Kevin De Bruyne,Right Wing,wide_player,3.365938,3.840621,2.804948,0.647296
59,Christian Dannemann Eriksen,Left Wing,wide_player,3.164432,2.188486,2.543375,0.680205
137,Dušan Tadić,Left Wing,wide_player,2.951972,5.137198,1.571829,0.690071
211,David Josué Jiménez Silva,Center Attacking Midfield,attacking_midfielder,2.937044,2.055931,1.909079,0.440557
129,Erik Lamela,Right Wing,wide_player,2.656228,1.328114,1.180546,0.516489
267,Santiago Cazorla González,Left Defensive Midfield,defensive_midfielder,2.629345,1.550639,2.696764,1.348382
79,Willian Borges da Silva,Right Wing,wide_player,2.620149,3.125237,3.030533,0.378817
140,Eden Hazard,Left Wing,wide_player,2.530824,1.985724,1.829981,0.467229


## 10. Building Ball-Carrying Features

### Objective

Use StatsBomb carry events to describe how players move the ball while retaining possession.

The following candidate features are calculated:

- Average carry length.
- Average forward progression per carry.
- Progressive carries.

A carry is considered progressive when its end location is at least 25% closer to the opponent's goal than its start location. The same spatial rule used for progressive passes is therefore applied consistently.

In [1953]:
required_carry_columns = [
    "player_id",
    "player",
    "is_carry",
    "location",
    "carry_end_location",
]

missing_carry_columns = [
    column
    for column in required_carry_columns
    if column not in available_event_columns
]

assert not missing_carry_columns

print(
    "Carry source columns "
    "are available."
)

Carry source columns are available.


In [1954]:
carry_features = duckdb.sql(f"""
    WITH carry_locations AS (
        SELECT
            CAST(player_id AS BIGINT)
                AS player_id,

            player,

            carry_end_location[1]
            - location[1]
                AS delta_x,

            carry_end_location[2]
            - location[2]
                AS delta_y,

            SQRT(
                POWER(
                    carry_end_location[1]
                    - location[1],
                    2
                )
                +
                POWER(
                    carry_end_location[2]
                    - location[2],
                    2
                )
            )
                AS carry_length,

            SQRT(
                POWER(
                    120.0 - location[1],
                    2
                )
                +
                POWER(
                    40.0 - location[2],
                    2
                )
            )
                AS start_goal_distance,

            SQRT(
                POWER(
                    120.0
                    - carry_end_location[1],
                    2
                )
                +
                POWER(
                    40.0
                    - carry_end_location[2],
                    2
                )
            )
                AS end_goal_distance

        FROM read_parquet(
            '{EVENTS_GLOB}',
            union_by_name = true
        )

        WHERE
            is_carry
            AND player_id IS NOT NULL
            AND location IS NOT NULL
            AND carry_end_location IS NOT NULL
    )

    SELECT
        player_id,

        ANY_VALUE(player)
            AS player_name_events,

        COUNT(*)
            AS spatial_carries,

        AVG(carry_length)
            AS average_carry_length,

        AVG(delta_x)
            AS average_carry_progression,

        SUM(
            CASE
                WHEN
                    end_goal_distance
                    <= {PROGRESSIVE_DISTANCE_RATIO}
                    * start_goal_distance
                THEN 1
                ELSE 0
            END
        )
            AS progressive_carries

    FROM carry_locations

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1955]:
assert (
    carry_features[
        "spatial_carries"
    ].sum()
    == player_features[
        "carries"
    ].sum()
)

assert (
    carry_features[
        "progressive_carries"
    ]
    <= carry_features[
        "spatial_carries"
    ]
).all()

assert (
    carry_features["player_id"]
    .is_unique
)

assert (
    carry_features[
        "average_carry_progression"
    ].abs()
    <= carry_features[
        "average_carry_length"
    ] + 1e-9
).all()

print(
    "Carry features calculated "
    "and validated."
)

Carry features calculated and validated.


In [1956]:
carry_features.sort_values(
    "progressive_carries",
    ascending=False,
).head(15)

,player_id,player_name_events,spatial_carries,average_carry_length,average_carry_progression,progressive_carries
453,20039,Anthony Martial,1193,8.080128,3.711567,207.0
181,3621,Eden Hazard,1468,8.411338,4.150545,205.0
114,3472,Willian Borges da Silva,1484,8.295257,3.574798,201.0
133,3507,Wilfried Zaha,1085,9.550438,6.227558,197.0
225,3814,Riyad Mahrez,1282,7.932437,4.271295,191.0
184,3629,Sadio Mané,1228,8.425847,3.791124,187.0
24,3091,Moussa Sissoko,1418,8.340246,5.284415,170.0
333,6821,Jesús Navas González,1320,7.222331,3.369242,168.0
264,4275,Ross Barkley,1858,6.954561,2.981485,157.0
219,3726,Gerard Deulofeu Lázaro,734,8.319473,4.686240,150.0


### Merging Ball-Carrying Features

Ball-carrying features are merged into the player feature table.

Average carry length and average carry progression describe the typical characteristics of an individual carry.

Progressive-carry totals are normalized per 90 minutes because they depend on playing time.

In [1957]:
carry_columns_to_merge = [
    "player_id",
    "average_carry_length",
    "average_carry_progression",
    "progressive_carries",
]

assert (
    carry_features["player_id"]
    .is_unique
)

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        carry_features[
            carry_columns_to_merge
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(player_features) == rows_before_merge

In [1958]:
player_features["progressive_carries"] = (
    player_features["progressive_carries"]
    .fillna(0)
    .astype("int32")
)

In [1959]:
player_features[
    "progressive_carries_per90"
] = (
    player_features[
        "progressive_carries"
    ]
    / player_features["minutes_played"]
    * 90
)

In [1960]:
assert (
    player_features["progressive_carries"]
    <= player_features["carries"]
).all()

assert (
    player_features[
        "progressive_carries_per90"
    ] >= 0
).all()

eligible_carry_missing = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        [
            "average_carry_length",
            "average_carry_progression",
        ],
    ]
    .isna()
    .sum()
)

print(
    "Carry features merged "
    "and validated."
)

eligible_carry_missing

Carry features merged and validated.


average_carry_length         0
average_carry_progression    0
dtype: int64

In [1961]:
carry_leaders_by_role = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ],
        [
            "player_name",
            "primary_position",
            "role",
            "carries_per90",
            "average_carry_length",
            "average_carry_progression",
            "progressive_carries_per90",
        ],
    ]
    .sort_values(
        [
            "role",
            "progressive_carries_per90",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "role",
        group_keys=False,
    )
    .head(3)
)

carry_leaders_by_role

,player_name,primary_position,role,carries_per90,average_carry_length,average_carry_progression,progressive_carries_per90
97,Sadio Mané,Center Attacking Midfield,attacking_midfielder,40.966967,8.425847,3.791124,6.238455
211,David Josué Jiménez Silva,Center Attacking Midfield,attacking_midfielder,63.684902,6.079074,2.534819,4.601369
43,Ross Barkley,Center Attacking Midfield,attacking_midfielder,52.596208,6.954561,2.981485,4.444351
191,Manuel Lanzini,Left Center Midfield,central_midfielder,51.129545,7.294020,3.404278,4.374720
105,Emre Can,Left Center Midfield,central_midfielder,48.276568,7.001698,3.343848,3.453259
225,Harry Arter,Left Center Midfield,central_midfielder,47.344756,6.172080,3.172514,2.803656
135,José Ramiro Funes Mori,Left Center Back,centre_back,36.226495,7.818021,5.227713,2.099533
69,Laurent Koscielny,Left Center Back,centre_back,36.676752,7.359279,4.003554,1.758059
32,Daley Blind,Left Center Back,centre_back,41.898942,6.802219,4.086300,1.663757
269,Gilbert Gianelli Imbula Wanga,Left Defensive Midfield,defensive_midfielder,49.669426,8.346465,4.082853,4.496820


## 11. Building Shooting-Profile Features

### Objective

Separate shooting volume from average shot quality.

Penalty shots are excluded because they are structurally different from open-play and non-penalty shooting opportunities.

The following features are created:

- Non-penalty shots per 90 minutes.
- Average non-penalty expected goals per shot.

Together, these variables describe how frequently a player shoots and the average quality of the chances selected.

In [1962]:
required_shot_columns = [
    "player_id",
    "player",
    "is_shot",
    "is_penalty",
    "shot_statsbomb_xg",
]

missing_shot_columns = [
    column
    for column in required_shot_columns
    if column not in available_event_columns
]

assert not missing_shot_columns

print(
    "Shooting source columns "
    "are available."
)

Shooting source columns are available.


In [1963]:
shooting_profile = duckdb.sql(f"""
    SELECT
        CAST(player_id AS BIGINT)
            AS player_id,

        ANY_VALUE(player)
            AS player_name_events,

        COUNT(*)
            AS event_shots,

        SUM(
            CASE
                WHEN NOT is_penalty THEN 1
                ELSE 0
            END
        )
            AS non_penalty_shots,

        SUM(
            CASE
                WHEN NOT is_penalty
                THEN shot_statsbomb_xg
                ELSE 0
            END
        )
            AS event_non_penalty_xg

    FROM read_parquet(
        '{EVENTS_GLOB}',
        union_by_name = true
    )

    WHERE
        is_shot
        AND player_id IS NOT NULL

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1964]:
assert (
    shooting_profile[
        "event_shots"
    ].sum()
    == player_features[
        "shots"
    ].sum()
)

non_penalty_xg_difference = abs(
    shooting_profile[
        "event_non_penalty_xg"
    ].sum()
    - player_features[
        "non_penalty_xg"
    ].sum()
)

assert non_penalty_xg_difference < 1e-6

assert (
    shooting_profile[
        "non_penalty_shots"
    ]
    <= shooting_profile[
        "event_shots"
    ]
).all()

assert (
    shooting_profile["player_id"]
    .is_unique
)

print(
    "Non-penalty shooting totals "
    "validated."
)

Non-penalty shooting totals validated.


In [1965]:
shooting_profile.sort_values(
    "non_penalty_shots",
    ascending=False,
).head(15)

,player_id,player_name_events,event_shots,non_penalty_shots,event_non_penalty_xg
340,10955,Harry Kane,158,153.0,17.717991
46,3289,Romelu Lukaku Menama,116,114.0,16.590158
33,3237,Sergio Leonel Agüero del Castillo,118,113.0,13.755574
343,10960,Jamie Vardy,118,112.0,17.597685
120,3501,Philippe Coutinho Correia,111,111.0,7.128246
82,3385,Alexis Alejandro Sánchez Sánchez,111,111.0,11.263074
271,5458,Odion Jude Ighalo,111,111.0,16.919278
155,3604,Olivier Giroud,103,102.0,12.234203
5,3043,Christian Dannemann Eriksen,99,99.0,5.109807
235,4275,Ross Barkley,95,93.0,6.117943


### Merging Shooting-Profile Features

Non-penalty shot totals are merged into the player feature table.

The totals are used to calculate:

- Non-penalty shots per 90 minutes.
- Average non-penalty expected goals per shot.

Players with no non-penalty shots retain an undefined average shot quality rather than receiving an artificial value of zero.

In [1966]:
shooting_columns_to_merge = [
    "player_id",
    "non_penalty_shots",
]

assert (
    shooting_profile["player_id"]
    .is_unique
)

rows_before_merge = len(
    player_features
)

player_features = (
    player_features
    .merge(
        shooting_profile[
            shooting_columns_to_merge
        ],
        on="player_id",
        how="left",
        validate="one_to_one",
    )
)

assert len(player_features) == rows_before_merge

In [1967]:
player_features["non_penalty_shots"] = (
    player_features["non_penalty_shots"]
    .fillna(0)
    .astype("int32")
)

In [1968]:
player_features[
    "non_penalty_shots_per90"
] = (
    player_features[
        "non_penalty_shots"
    ]
    / player_features["minutes_played"]
    * 90
)

In [1969]:
player_features[
    "average_non_penalty_xg_per_shot"
] = calculate_rate(
    player_features["non_penalty_xg"],
    player_features["non_penalty_shots"],
)

In [1970]:
assert (
    player_features["non_penalty_shots"]
    <= player_features["shots"]
).all()

assert (
    player_features[
        "non_penalty_shots_per90"
    ] >= 0
).all()

available_shot_quality = (
    player_features[
        "average_non_penalty_xg_per_shot"
    ]
    .dropna()
)

assert available_shot_quality.between(
    0,
    1,
).all()

print(
    "Shooting-profile features merged "
    "and validated."
)

Shooting-profile features merged and validated.


In [1971]:
eligible_shot_quality = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ]
    ]
)

missing_shot_quality_by_role = (
    eligible_shot_quality
    .groupby("role")[
        "average_non_penalty_xg_per_shot"
    ]
    .apply(
        lambda values: values.isna().sum()
    )
)

missing_shot_quality_by_role

role
attacking_midfielder     0
central_midfielder       0
centre_back              1
defensive_midfielder     0
forward                  0
full_back                1
goalkeeper              26
wide_player              0
Name: average_non_penalty_xg_per_shot, dtype: int64

In [1972]:
attacking_roles = [
    "attacking_midfielder",
    "wide_player",
    "forward",
]

shooting_leaders_by_role = (
    player_features.loc[
        player_features[
            "eligible_for_similarity"
        ]
        & player_features["role"].isin(
            attacking_roles
        ),
        [
            "player_name",
            "primary_position",
            "role",
            "non_penalty_shots_per90",
            "non_penalty_xg_per90",
            "average_non_penalty_xg_per_shot",
        ],
    ]
    .sort_values(
        [
            "role",
            "non_penalty_shots_per90",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "role",
        group_keys=False,
    )
    .head(5)
)

shooting_leaders_by_role

,player_name,primary_position,role,non_penalty_shots_per90,non_penalty_xg_per90,average_non_penalty_xg_per_shot
177,Roberto Firmino Barbosa de Oliveira,Center Attacking Midfield,attacking_midfielder,2.859983,0.323475,0.113104
97,Sadio Mané,Center Attacking Midfield,attacking_midfielder,2.835661,0.375928,0.132572
43,Ross Barkley,Center Attacking Midfield,attacking_midfielder,2.632641,0.173187,0.065784
116,Bamidele Alli,Center Attacking Midfield,attacking_midfielder,2.589078,0.297867,0.115047
60,Gylfi Þór Sigurðsson,Center Attacking Midfield,attacking_midfielder,2.543905,0.152187,0.059824
317,Daniel Andre Sturridge,Center Forward,forward,4.225834,0.451190,0.106769
130,Sergio Leonel Agüero del Castillo,Center Forward,forward,4.197536,0.510969,0.121731
270,Wilfried Guemiand Bony,Center Forward,forward,4.175688,0.452156,0.108283
9,Harry Kane,Center Forward,forward,3.949350,0.457350,0.115804
114,Olivier Giroud,Center Forward,forward,3.602590,0.432106,0.119943


## 12. Building Spatial Pressing Features

### Objective

Complement defensive event volumes with information about where pressures occur on the pitch.

Two spatial pressing features are calculated:

- Average pressure height.
- High pressures in the attacking third.

The attacking third begins at `x = 80` on the 120-unit StatsBomb pitch.

In [1973]:
ATTACKING_THIRD_X = 80.0

In [1974]:
required_pressure_columns = [
    "player_id",
    "player",
    "is_pressure",
    "location",
]

missing_pressure_columns = [
    column
    for column in required_pressure_columns
    if column not in available_event_columns
]

assert not missing_pressure_columns

print(
    "Pressure source columns "
    "are available."
)

Pressure source columns are available.


In [1975]:
pressure_features = duckdb.sql(f"""
    SELECT
        CAST(player_id AS BIGINT)
            AS player_id,

        ANY_VALUE(player)
            AS player_name_events,

        COUNT(*)
            AS spatial_pressures,

        AVG(location[1])
            AS average_pressure_height,

        SUM(
            CASE
                WHEN location[1]
                    >= {ATTACKING_THIRD_X}
                THEN 1
                ELSE 0
            END
        )
            AS high_pressures

    FROM read_parquet(
        '{EVENTS_GLOB}',
        union_by_name = true
    )

    WHERE
        is_pressure
        AND player_id IS NOT NULL
        AND location IS NOT NULL

    GROUP BY
        player_id

    ORDER BY
        player_id
""").df()

In [1976]:
assert (
    pressure_features[
        "spatial_pressures"
    ].sum()
    == player_features[
        "pressures"
    ].sum()
)

assert (
    pressure_features[
        "high_pressures"
    ]
    <= pressure_features[
        "spatial_pressures"
    ]
).all()

assert (
    pressure_features[
        "average_pressure_height"
    ].between(
        0,
        120,
    )
).all()

assert (
    pressure_features["player_id"]
    .is_unique
)

print(
    "Spatial pressure features "
    "calculated and validated."
)

Spatial pressure features calculated and validated.


In [1977]:
pressure_features.sort_values(
    "high_pressures",
    ascending=False,
).head(15)

,player_id,player_name_events,spatial_pressures,average_pressure_height,high_pressures
391,10960,Jamie Vardy,493,80.156998,271.0
245,4041,José Salomón Rondón Giménez,517,78.782398,258.0
229,3944,Gylfi Þór Sigurðsson,743,67.718035,249.0
388,10955,Harry Kane,491,77.748269,244.0
81,3346,Joshua King,593,74.287015,233.0
101,3450,Jordan Ayew,699,66.600858,227.0
298,5458,Odion Jude Ighalo,498,74.293173,223.0
102,3454,Troy Deeney,609,70.099507,221.0
25,3094,Bamidele Alli,689,64.270682,211.0
438,20039,Anthony Martial,372,73.493548,184.0


### Merging Spatial Pressing Features

The spatial pressure features are merged into the main player table using
`player_id`.

Only the new information is retained:

- `average_pressure_height`;
- `high_pressures`;
- `high_pressures_per90`.

The existing `pressures` and `pressures_per90` columns already represent total
pressing volume, so `spatial_pressures` would be redundant.

In [1978]:
rows_before_pressure_merge = len(player_features)

player_features = player_features.merge(
    pressure_features[
        [
            "player_id",
            "average_pressure_height",
            "high_pressures",
        ]
    ],
    on="player_id",
    how="left",
    validate="one_to_one",
)

assert len(player_features) == rows_before_pressure_merge
assert player_features["player_id"].is_unique

print(
    "Pressure features merged:",
    player_features.shape,
)

Pressure features merged: (550, 69)


In [1979]:
player_features["high_pressures"] = (
    player_features["high_pressures"]
    .fillna(0)
    .astype("int32")
)

In [1980]:
player_features["high_pressures_per90"] = (
    player_features["high_pressures"]
    / player_features["minutes_played"]
    * 90
)

In [1981]:
assert (
    player_features["high_pressures"]
    <= player_features["pressures"]
).all()

assert (
    player_features["high_pressures_per90"] >= 0
).all()

assert player_features[
    "high_pressures_per90"
].notna().all()

available_pressure_heights = player_features[
    "average_pressure_height"
].dropna()

assert available_pressure_heights.between(
    0,
    120,
    inclusive="both",
).all()

print(
    "Spatial pressure features validated."
)

Spatial pressure features validated.


In [1982]:
eligible_pressure_missing = (
    player_features.loc[
        player_features["eligible_for_similarity"]
    ]
    .groupby("role")
    .agg(
        eligible_players=(
            "player_id",
            "size",
        ),
        missing_pressure_height=(
            "average_pressure_height",
            lambda values: values.isna().sum(),
        ),
    )
    .sort_values(
        "missing_pressure_height",
        ascending=False,
    )
)

eligible_pressure_missing

,eligible_players,missing_pressure_height
role,,
goalkeeper,27,1
attacking_midfielder,17,0
central_midfielder,31,0
centre_back,60,0
defensive_midfielder,37,0
forward,36,0
full_back,59,0
wide_player,60,0


In [1983]:
pressure_leaders_by_role = (
    player_features.loc[
        player_features["eligible_for_similarity"],
        [
            "player_name",
            "primary_position",
            "role",
            "pressures_per90",
            "average_pressure_height",
            "high_pressures_per90",
            "counterpressure_share",
        ],
    ]
    .sort_values(
        [
            "role",
            "high_pressures_per90",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "role",
        group_keys=False,
    )
    .head(3)
)

pressure_leaders_by_role

,player_name,primary_position,role,pressures_per90,average_pressure_height,high_pressures_per90,counterpressure_share
177,Roberto Firmino Barbosa de Oliveira,Center Attacking Midfield,attacking_midfielder,27.015840,66.205700,7.567955,0.258958
116,Bamidele Alli,Center Attacking Midfield,attacking_midfielder,24.436636,64.270682,7.483498,0.239478
60,Gylfi Þór Sigurðsson,Center Attacking Midfield,attacking_midfielder,21.978154,67.718035,7.365492,0.282638
115,James Philip Milner,Right Center Midfield,central_midfielder,22.548749,60.398899,5.459917,0.248428
111,Dan Gosling,Right Center Midfield,central_midfielder,22.632255,56.484211,4.169100,0.232198
299,Marouane Fellaini-Bakkioui,Right Center Midfield,central_midfielder,18.202338,57.919196,3.900501,0.205357
2,Simon Francis,Right Center Back,centre_back,10.098657,41.293548,1.027407,0.186104
179,James Tomkins,Right Center Back,centre_back,11.928206,38.962731,0.924326,0.173432
123,Geoff Cameron,Right Center Back,centre_back,14.542877,39.870175,0.874760,0.180451
91,Aaron Ramsey,Left Defensive Midfield,defensive_midfielder,22.855753,55.675000,4.203357,0.234195


## 13. Selecting Role-Specific Candidate Features

### Objective

Player similarity should be evaluated within broad roles because the same
metric does not have the same importance for every position.

For example:

- shooting features are important for forwards, but not for goalkeepers;
- clearances and blocks are useful for centre-backs;
- crosses are especially informative for full-backs and wide players;
- passing-distribution features are essential for goalkeepers and midfielders.

The following feature sets are candidate definitions for the similarity
engine. They will be validated for availability, missing values and variance
before any modelling step.

In [1984]:
role_feature_sets = {
    "goalkeeper": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "average_completed_pass_length",
        "average_completed_pass_progression",
        "progressive_passes_per90",
    ],

    "centre_back": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "average_completed_pass_length",
        "average_completed_pass_progression",
        "progressive_passes_per90",
        "carries_per90",
        "progressive_carries_per90",
        "pressures_per90",
        "interceptions_per90",
        "ball_recoveries_per90",
        "clearances_per90",
        "blocks_per90",
    ],

    "full_back": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "progressive_passes_per90",
        "key_passes_per90",
        "crosses_per90",
        "carries_per90",
        "progressive_carries_per90",
        "dribbles_attempted_per90",
        "dribble_completion_rate",
        "pressures_per90",
        "high_pressures_per90",
        "interceptions_per90",
        "ball_recoveries_per90",
    ],

    "defensive_midfielder": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "average_completed_pass_length",
        "average_completed_pass_progression",
        "progressive_passes_per90",
        "key_passes_per90",
        "switches_per90",
        "carries_per90",
        "progressive_carries_per90",
        "pressures_per90",
        "counterpressure_share",
        "interceptions_per90",
        "ball_recoveries_per90",
    ],

    "central_midfielder": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "average_completed_pass_progression",
        "progressive_passes_per90",
        "key_passes_per90",
        "switches_per90",
        "through_balls_per90",
        "carries_per90",
        "progressive_carries_per90",
        "dribbles_attempted_per90",
        "dribble_completion_rate",
        "pressures_per90",
        "counterpressure_share",
        "interceptions_per90",
        "ball_recoveries_per90",
    ],

    "wide_player": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "progressive_passes_per90",
        "key_passes_per90",
        "crosses_per90",
        "through_balls_per90",
        "carries_per90",
        "average_carry_length",
        "progressive_carries_per90",
        "dribbles_attempted_per90",
        "dribble_completion_rate",
        "non_penalty_shots_per90",
        "average_non_penalty_xg_per_shot",
        "pressures_per90",
        "high_pressures_per90",
    ],

    "attacking_midfielder": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "progressive_passes_per90",
        "key_passes_per90",
        "switches_per90",
        "through_balls_per90",
        "carries_per90",
        "average_carry_length",
        "progressive_carries_per90",
        "dribbles_attempted_per90",
        "dribble_completion_rate",
        "non_penalty_shots_per90",
        "average_non_penalty_xg_per_shot",
        "pressures_per90",
        "high_pressures_per90",
    ],

    "forward": [
        "passes_attempted_per90",
        "pass_completion_rate",
        "key_passes_per90",
        "carries_per90",
        "progressive_carries_per90",
        "dribbles_attempted_per90",
        "dribble_completion_rate",
        "non_penalty_shots_per90",
        "average_non_penalty_xg_per_shot",
        "pressures_per90",
        "high_pressures_per90",
    ],
}

In [1985]:
selected_feature_columns = {
    feature
    for features in role_feature_sets.values()
    for feature in features
}

missing_selected_features = (
    selected_feature_columns
    - set(player_features.columns)
)

assert not missing_selected_features, (
    "Missing selected features: "
    f"{sorted(missing_selected_features)}"
)

print(
    f"{len(selected_feature_columns)} unique "
    "candidate features are available."
)

23 unique candidate features are available.


In [1986]:
available_roles = set(
    player_features["role"].dropna().unique()
)

configured_roles = set(
    role_feature_sets
)

assert configured_roles == available_roles, {
    "missing_role_configurations": (
        available_roles - configured_roles
    ),
    "unknown_role_configurations": (
        configured_roles - available_roles
    ),
}

print("All player roles have a feature configuration.")

All player roles have a feature configuration.


In [1987]:
role_feature_validation_rows = []

for role, features in role_feature_sets.items():
    role_players = player_features.loc[
        (
            player_features["eligible_for_similarity"]
            & player_features["role"].eq(role)
        ),
        features,
    ]

    missing_values = int(
        role_players.isna().sum().sum()
    )

    constant_features = [
        feature
        for feature in features
        if role_players[feature].nunique(
            dropna=True
        ) <= 1
    ]

    role_feature_validation_rows.append(
        {
            "role": role,
            "eligible_players": len(role_players),
            "selected_features": len(features),
            "missing_values": missing_values,
            "constant_features": constant_features,
        }
    )

role_feature_validation = pd.DataFrame(
    role_feature_validation_rows
)

role_feature_validation

,role,eligible_players,selected_features,missing_values,constant_features
0,goalkeeper,27,5,0,[]
1,centre_back,60,12,0,[]
2,full_back,59,13,0,[]
3,defensive_midfielder,37,13,0,[]
4,central_midfielder,31,15,0,[]
5,wide_player,60,15,0,[]
6,attacking_midfielder,17,15,0,[]
7,forward,36,11,0,[]


## 14. Reviewing Feature Redundancy

### Objective

A similarity distance can unintentionally give excessive importance to one
aspect of play when several selected features measure nearly the same thing.

For example, total pressures and high pressures may be strongly correlated for
some roles.

This section identifies highly correlated feature pairs within each role.
The results are reviewed manually because correlation alone is not sufficient
to remove a feature: two correlated metrics can still represent different
football concepts.

In [1988]:
assert (
    role_feature_validation["missing_values"] == 0
).all()

assert role_feature_validation[
    "constant_features"
].apply(len).eq(0).all()

print(
    "All candidate role features are complete "
    "and non-constant."
)

All candidate role features are complete and non-constant.


In [1989]:
CORRELATION_REVIEW_THRESHOLD = 0.85

high_correlation_rows = []

for role, features in role_feature_sets.items():
    role_players = player_features.loc[
        (
            player_features["eligible_for_similarity"]
            & player_features["role"].eq(role)
        ),
        features,
    ]

    correlation_matrix = (
        role_players
        .corr()
    )

    for left_index, left_feature in enumerate(
        features
    ):
        for right_feature in features[
            left_index + 1:
        ]:
            correlation = correlation_matrix.loc[
                left_feature,
                right_feature,
            ]

            if (
                pd.notna(correlation)
                and abs(correlation)
                >= CORRELATION_REVIEW_THRESHOLD
            ):
                high_correlation_rows.append(
                    {
                        "role": role,
                        "feature_1": left_feature,
                        "feature_2": right_feature,
                        "correlation": correlation,
                        "absolute_correlation": abs(
                            correlation
                        ),
                    }
                )

high_role_correlations = pd.DataFrame(
    high_correlation_rows,
    columns=[
        "role",
        "feature_1",
        "feature_2",
        "correlation",
        "absolute_correlation",
    ],
)

high_role_correlations = (
    high_role_correlations
    .sort_values(
        [
            "absolute_correlation",
            "role",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

high_role_correlations

,role,feature_1,feature_2,correlation,absolute_correlation
0,goalkeeper,average_completed_pass_length,average_completed_pass_progression,0.993047,0.993047
1,centre_back,passes_attempted_per90,carries_per90,0.974083,0.974083
2,defensive_midfielder,passes_attempted_per90,carries_per90,0.932073,0.932073
3,central_midfielder,passes_attempted_per90,carries_per90,0.913731,0.913731
4,attacking_midfielder,passes_attempted_per90,carries_per90,0.909060,0.909060
5,attacking_midfielder,pressures_per90,high_pressures_per90,0.895182,0.895182
6,full_back,passes_attempted_per90,carries_per90,0.871516,0.871516


## 15. Finalizing Role-Specific Feature Sets

### Redundancy decisions

The correlation review identified a small number of highly correlated feature
pairs.

The following adjustments are applied:

- goalkeeper pass progression is removed because it is almost identical to
  completed pass length;
- total carry volume is removed for roles where it mainly duplicates passing
  involvement;
- high-pressure volume is removed for attacking midfielders because it closely
  duplicates total pressure volume in a small comparison group.

Progressive carrying metrics are retained because they describe ball
advancement rather than general possession involvement.

In [1990]:
role_feature_sets["goalkeeper"] = [
    feature
    for feature in role_feature_sets["goalkeeper"]
    if feature != "average_completed_pass_progression"
]

roles_without_total_carry_volume = [
    "centre_back",
    "full_back",
    "defensive_midfielder",
    "central_midfielder",
    "attacking_midfielder",
]

for role in roles_without_total_carry_volume:
    role_feature_sets[role] = [
        feature
        for feature in role_feature_sets[role]
        if feature != "carries_per90"
    ]

role_feature_sets["attacking_midfielder"] = [
    feature
    for feature in role_feature_sets[
        "attacking_midfielder"
    ]
    if feature != "high_pressures_per90"
]

In [1991]:
final_role_feature_summary = pd.DataFrame(
    [
        {
            "role": role,
            "eligible_players": int(
                (
                    player_features[
                        "eligible_for_similarity"
                    ]
                    & player_features["role"].eq(role)
                ).sum()
            ),
            "selected_features": len(features),
            "features": features,
        }
        for role, features in role_feature_sets.items()
    ]
)

final_role_feature_summary

,role,eligible_players,selected_features,features
0,goalkeeper,27,4,"[passes_attempted_per90, pass_completion_rate,..."
1,centre_back,60,11,"[passes_attempted_per90, pass_completion_rate,..."
2,full_back,59,12,"[passes_attempted_per90, pass_completion_rate,..."
3,defensive_midfielder,37,12,"[passes_attempted_per90, pass_completion_rate,..."
4,central_midfielder,31,14,"[passes_attempted_per90, pass_completion_rate,..."
5,wide_player,60,15,"[passes_attempted_per90, pass_completion_rate,..."
6,attacking_midfielder,17,13,"[passes_attempted_per90, pass_completion_rate,..."
7,forward,36,11,"[passes_attempted_per90, pass_completion_rate,..."


In [1992]:
def find_high_role_correlations(
    data,
    feature_sets,
    threshold=0.85,
):
    correlation_rows = []

    for role, features in feature_sets.items():
        role_data = data.loc[
            (
                data["eligible_for_similarity"]
                & data["role"].eq(role)
            ),
            features,
        ]

        correlation_matrix = role_data.corr()

        for left_index, left_feature in enumerate(
            features
        ):
            for right_feature in features[
                left_index + 1:
            ]:
                correlation = correlation_matrix.loc[
                    left_feature,
                    right_feature,
                ]

                if (
                    pd.notna(correlation)
                    and abs(correlation) >= threshold
                ):
                    correlation_rows.append(
                        {
                            "role": role,
                            "feature_1": left_feature,
                            "feature_2": right_feature,
                            "correlation": correlation,
                            "absolute_correlation": abs(
                                correlation
                            ),
                        }
                    )

    result = pd.DataFrame(
        correlation_rows,
        columns=[
            "role",
            "feature_1",
            "feature_2",
            "correlation",
            "absolute_correlation",
        ],
    )

    return result.sort_values(
        "absolute_correlation",
        ascending=False,
    ).reset_index(drop=True)

In [1993]:
remaining_high_correlations = (
    find_high_role_correlations(
        data=player_features,
        feature_sets=role_feature_sets,
        threshold=CORRELATION_REVIEW_THRESHOLD,
    )
)

remaining_high_correlations

,role,feature_1,feature_2,correlation,absolute_correlation


In [1994]:
assert remaining_high_correlations.empty

assert all(
    len(features) > 0
    for features in role_feature_sets.values()
)

print(
    "Final role-specific feature sets validated."
)

Final role-specific feature sets validated.


## 16. Final Validation and Export

### Objective

The feature-engineering dataset is validated before being exported.

The exported table retains every player and includes the
`eligible_for_similarity` flag. This preserves the complete season data while
allowing later similarity calculations to use only players with sufficient
playing time.

The role-specific feature configuration is also exported so that future
similarity steps use exactly the feature definitions validated in this
notebook.

No feature scaling or similarity modelling is performed in this phase.

In [1995]:
assert len(player_features) == len(player_season)

assert player_features["player_id"].is_unique

assert player_features["role"].notna().all()

assert (
    player_features["eligible_for_similarity"]
    == (
        player_features["minutes_played"]
        >= MINIMUM_MINUTES
    )
).all()

assert set(role_feature_sets) == set(
    player_features["role"].unique()
)

print(
    "General dataset structure validated."
)

General dataset structure validated.


In [1996]:
final_selected_feature_columns = {
    feature
    for features in role_feature_sets.values()
    for feature in features
}

assert final_selected_feature_columns.issubset(
    player_features.columns
)

for role, features in role_feature_sets.items():
    assert len(features) == len(set(features))

    role_players = player_features.loc[
        (
            player_features["eligible_for_similarity"]
            & player_features["role"].eq(role)
        ),
        features,
    ]

    assert len(role_players) > 0

    assert not role_players.isna().any().any()

    assert not role_players.isin(
        [
            float("inf"),
            float("-inf"),
        ]
    ).any().any()

    assert all(
        pd.api.types.is_numeric_dtype(
            role_players[feature]
        )
        for feature in features
    )

print(
    f"{len(final_selected_feature_columns)} unique "
    "features validated across all roles."
)

23 unique features validated across all roles.


In [1997]:
final_dataset_summary = pd.DataFrame(
    {
        "metric": [
            "all_players",
            "eligible_players",
            "ineligible_players",
            "roles",
            "dataset_columns",
            "selected_feature_columns",
        ],
        "value": [
            len(player_features),
            int(
                player_features[
                    "eligible_for_similarity"
                ].sum()
            ),
            int(
                (
                    ~player_features[
                        "eligible_for_similarity"
                    ]
                ).sum()
            ),
            player_features["role"].nunique(),
            len(player_features.columns),
            len(final_selected_feature_columns),
        ],
    }
)

final_dataset_summary

,metric,value
0,all_players,550
1,eligible_players,327
2,ineligible_players,223
3,roles,8
4,dataset_columns,70
5,selected_feature_columns,23


In [1998]:
import json


PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PLAYER_FEATURES_PATH = (
    PROCESSED_DATA_DIR
    / "player_features.parquet"
)

ROLE_FEATURE_SETS_PATH = (
    PROCESSED_DATA_DIR
    / "role_feature_sets.json"
)


PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

player_features.to_parquet(
    PLAYER_FEATURES_PATH,
    index=False,
)

with ROLE_FEATURE_SETS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        role_feature_sets,
        file,
        indent=4,
        ensure_ascii=False,
    )

print(f"Player features: {PLAYER_FEATURES_PATH}")
print(f"Role configuration: {ROLE_FEATURE_SETS_PATH}")

Player features: /Users/bilal/Desktop/football-player-similarity/data/processed/player_features.parquet
Role configuration: /Users/bilal/Desktop/football-player-similarity/data/processed/role_feature_sets.json


In [1999]:
saved_player_features = pd.read_parquet(
    PLAYER_FEATURES_PATH
)

with ROLE_FEATURE_SETS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    saved_role_feature_sets = json.load(file)


pd.testing.assert_frame_equal(
    saved_player_features,
    player_features.reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

assert saved_role_feature_sets == role_feature_sets

print(
    "Export completed and round-trip validation passed."
)

Export completed and round-trip validation passed.


# Test function

In [2000]:
import sys


PROJECT_ROOT_STRING = str(PROJECT_ROOT)

if PROJECT_ROOT_STRING not in sys.path:
    sys.path.insert(
        0,
        PROJECT_ROOT_STRING,
    )

print(f"Import root: {PROJECT_ROOT_STRING}")

Import root: /Users/bilal/Desktop/football-player-similarity


In [2001]:
from src.features import (
    build_base_player_features,
)


reproduced_base_features = (
    build_base_player_features(
        player_season
    )
)

print(
    "Shape:",
    reproduced_base_features.shape,
)

print(
    "Eligible players:",
    reproduced_base_features[
        "eligible_for_similarity"
    ].sum(),
)

print(
    "Roles:",
    reproduced_base_features[
        "role"
    ].nunique(),
)

Shape: (550, 44)
Eligible players: 327
Roles: 8


In [2002]:
import importlib

import src.features as features_module


importlib.reload(features_module)

reproduced_passing_features = (
    features_module.build_passing_features(
        PROJECT_ROOT
        / "data"
        / "processed"
        / "events"
    )
)

print(
    "Players with passes:",
    len(reproduced_passing_features),
)

assert (
    reproduced_passing_features[
        "event_passes_attempted"
    ].sum()
    == player_season[
        "passes_attempted"
    ].sum()
)

assert (
    reproduced_passing_features[
        "event_passes_completed"
    ].sum()
    == player_season[
        "passes_completed"
    ].sum()
)

print(
    "Passing feature totals reproduced."
)


reproduced_passing_features.head()

Players with passes: 545
Passing feature totals reproduced.


,player_id,player_name_events,event_passes_attempted,event_passes_completed,average_pass_length,average_completed_pass_length,average_pass_progression,average_completed_pass_progression,completed_forward_distance,progressive_passes,key_passes,crosses,switches,through_balls
0,2956,Bertrand Isidore Traoré,137,115.0,16.157784,15.817100,-1.525547,-2.118261,389.3,11.0,5.0,4.0,1.0,1.0
1,2966,Stéphane Sessègnon,638,463.0,17.928795,16.499435,1.027900,-0.926782,2002.8,60.0,28.0,51.0,15.0,3.0
2,2988,Memphis Depay,751,547.0,17.105169,15.940834,0.133688,-2.375868,1886.9,65.0,12.0,63.0,32.0,5.0
3,3037,Mousa Sidi Yaya Dembélé,1383,1238.0,17.275203,17.227872,1.793203,0.495396,5846.9,65.0,26.0,5.0,17.0,3.0
4,3041,Ciaran Clark,584,445.0,24.177039,20.659406,11.987158,5.840000,3788.9,43.0,3.0,2.0,9.0,2.0


In [2003]:
import importlib

import src.features as features_module


importlib.reload(features_module)

reproduced_player_features = (
    features_module.build_player_features(
        player_season=player_season,
        events_directory=(
            PROJECT_ROOT
            / "data"
            / "processed"
            / "events"
        ),
    )
)

saved_player_features = pd.read_parquet(
    PLAYER_FEATURES_PATH
)

print(
    "Reproduced shape:",
    reproduced_player_features.shape,
)

print(
    "Notebook export shape:",
    saved_player_features.shape,
)

Reproduced shape: (550, 70)
Notebook export shape: (550, 70)


In [2004]:
missing_columns = (
    set(saved_player_features.columns)
    - set(reproduced_player_features.columns)
)

unexpected_columns = (
    set(reproduced_player_features.columns)
    - set(saved_player_features.columns)
)

assert not missing_columns, missing_columns
assert not unexpected_columns, unexpected_columns

print("Column sets match.")

Column sets match.


In [2005]:
saved_comparison = (
    saved_player_features
    .sort_values("player_id")
    .reset_index(drop=True)
)

reproduced_comparison = (
    reproduced_player_features[
        saved_player_features.columns
    ]
    .sort_values("player_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    saved_comparison,
    reproduced_comparison,
    check_dtype=False,
    check_exact=False,
    rtol=1e-8,
    atol=1e-8,
)

print(
    "The reusable pipeline exactly reproduces "
    "the notebook export."
)

AssertionError: DataFrame.iloc[:, 48] (column name="completed_forward_distance") are different

DataFrame.iloc[:, 48] (column name="completed_forward_distance") values are different (0.90909 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, ...]
[left]:  [389.3, 2002.8, 1886.9, 5846.9, 3788.9, 8660.6, 9489.3, 5631.0, 3771.0, 11761.6, 9645.0, 31.9, 12537.9, 7282.0, 5563.9, 2524.8, 11317.5, 2997.3, 1031.5, 300.4, 4084.4, 9260.6, 5048.8, 12124.1, 5734.1, 339.0, 4289.8, 15180.2, 7202.9, 18546.8, 5801.1, 48.9, 1473.3, 9607.3, 2728.3, 1934.8, 11722.0, 472.5, 20203.9, 8048.5, 630.5, 2807.7, 6592.5, 5645.9, 8239.9, 351.3, 3589.4, 6203.9, 2087.2, 1

In [2006]:
forward_distance_check = (
    saved_player_features[
        [
            "player_id",
            "player_name",
            "passes_attempted",
            "completed_forward_distance",
        ]
    ]
    .merge(
        reproduced_player_features[
            [
                "player_id",
                "completed_forward_distance",
            ]
        ],
        on="player_id",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_notebook",
            "_module",
        ),
    )
)

forward_distance_differences = (
    forward_distance_check.loc[
        (
            forward_distance_check[
                "completed_forward_distance_notebook"
            ].isna()
            & forward_distance_check[
                "completed_forward_distance_module"
            ].eq(0.0)
        )
    ]
)

forward_distance_differences

,player_id,player_name,passes_attempted,completed_forward_distance_notebook,completed_forward_distance_module
542,24120,Jamie Sterry,0,NaN,0.0
543,9635,Will Keane,0,NaN,0.0
546,9636,Rushian Hepburn-Murphy,0,NaN,0.0
548,4837,Josh Cullen,0,NaN,0.0
549,29384,James Weir,0,NaN,0.0


In [2007]:
assert (
    forward_distance_differences[
        "passes_attempted"
    ] == 0
).all()

print(
    "Players without passes:",
    len(forward_distance_differences),
)

Players without passes: 5


### Sum-Based Missing Values

Players without pass events do not appear in the event-level passing
aggregation.

For cumulative metrics, the correct value is zero rather than missing.
Therefore, `completed_forward_distance` is filled with zero before its per-90
version is calculated.

In [2008]:
player_features[
    "completed_forward_distance"
] = (
    player_features[
        "completed_forward_distance"
    ]
    .fillna(0.0)
)

player_features[
    "completed_forward_distance_per90"
] = (
    player_features[
        "completed_forward_distance"
    ]
    / player_features["minutes_played"]
    * 90
)

assert player_features[
    "completed_forward_distance"
].notna().all()

assert player_features[
    "completed_forward_distance_per90"
].notna().all()

In [2009]:
player_features.to_parquet(
    PLAYER_FEATURES_PATH,
    index=False,
)

saved_player_features = pd.read_parquet(
    PLAYER_FEATURES_PATH
)

print(
    "Corrected player features exported."
)

Corrected player features exported.


In [2010]:
saved_comparison = (
    saved_player_features
    .sort_values("player_id")
    .reset_index(drop=True)
)

reproduced_comparison = (
    reproduced_player_features[
        saved_player_features.columns
    ]
    .sort_values("player_id")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    saved_comparison,
    reproduced_comparison,
    check_dtype=False,
    check_exact=False,
    rtol=1e-8,
    atol=1e-8,
)

print(
    "The reusable pipeline exactly reproduces "
    "the corrected notebook export."
)

The reusable pipeline exactly reproduces the corrected notebook export.
